# 04 — Outer Loop: LLM-Guided Structure Search

Structure Descent outer loop:
1. **Diagnostics** — compute validation metrics and residual analysis
2. **LLM proposal** — send structured report to Claude; receive JSON proposal
3. **Validate** — parse and grammar-check the proposed structure S'
4. **Inner loop** — fit weights on S', compute Score(S', θ')
5. **Accept/reject** — keep S' only if Score(S', θ') > Score(S, θ)

This is Metropolis-Hastings over model structures, with the LLM as the proposal distribution.

**Requires**: `ANTHROPIC_API_KEY` in `.env`

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import json
from pathlib import Path
from dotenv import load_dotenv

from src.dsl import DSLStructure
from src.inner_loop import run_inner_loop
from src.outer_loop import generate_diagnostics, prompt_llm, apply_proposal, structure_descent_step, run_structure_descent
from src.evaluation import compute_metrics, compute_residuals, print_metrics
from src.checkpoint import Checkpoint

load_dotenv('../.env')

DATA_DIR = Path('../data')
ckpt = Checkpoint('../data')
print('Pipeline status:')
for stage, status in ckpt.status().items():
    print(f'  {stage:15s} {status}')

In [ ]:
# Load state from inner loop notebook
with open(DATA_DIR / 'train_features.pkl', 'rb') as f:
    feat_data = pickle.load(f)
with open(DATA_DIR / 'current_state.pkl', 'rb') as f:
    state = pickle.load(f)

features_list  = feat_data['features_list']
chosen_indices = feat_data['chosen_indices']
customer_ids   = feat_data['customer_ids']
categories     = feat_data['categories']

with open(DATA_DIR / 'train_choices.pkl', 'rb') as f:
    train_choices = pickle.load(f)
metadata = [e['metadata'] for e in train_choices[:len(features_list)]]

current_structure = DSLStructure.from_dict(state['structure'])
current_score     = state['score']
metrics           = state['metrics']
residuals         = state['residuals']

print(f'Starting: {current_structure}')
print(f'Score: {current_score:.2f}')

## Step 1: Inspect the diagnostic report

This is exactly what the LLM will receive as its proposal prompt.

In [ ]:
report = generate_diagnostics(current_structure, current_score, metrics, residuals)
print(report)

## Step 2: Define the fit function

Wraps the inner loop so the outer loop can call `fit_fn(structure)`.

In [ ]:
import json

# Load tuned hyperparameters if available; fall back to defaults silently.
TUNED_PATH = DATA_DIR / 'tuned_hyperparams.json'
if TUNED_PATH.exists():
    with open(TUNED_PATH) as f:
        _tuned = json.load(f)['best']
    HP_SIGMA = float(_tuned['sigma'])
    HP_TAU   = float(_tuned['tau'])
    HP_NU    = float(_tuned['nu'])
    print(f'Using tuned hyperparameters: sigma={HP_SIGMA}, tau={HP_TAU}, nu={HP_NU}')
else:
    HP_SIGMA, HP_TAU, HP_NU = 10.0, 1.0, 0.5
    print(f'No tuned_hyperparams.json — using defaults: sigma={HP_SIGMA}, tau={HP_TAU}, nu={HP_NU}')


def fit_fn(structure: DSLStructure):
    """Run inner loop and return (weights, posterior_score)."""
    # NOTE: in practice you should re-extract features for the new structure.
    # Here we reuse existing features for terms that overlap.
    # A full implementation would call extract_features() for the new term set.
    return run_inner_loop(
        structure, features_list, chosen_indices, customer_ids, categories,
        sigma=HP_SIGMA, tau=HP_TAU, nu=HP_NU
    )

print('fit_fn defined.')

## Step 3: Single outer-loop iteration (manual inspection)

In [ ]:
new_structure, new_score, accepted, proposal_detail = structure_descent_step(
    current_structure=current_structure,
    current_score=current_score,
    metrics=metrics,
    residuals=residuals,
    fit_fn=fit_fn,
    iteration=1,
)

if accepted:
    print(f'\n✓ New structure accepted: {new_structure}')
    current_structure = new_structure
    current_score = new_score
else:
    print(f'\n✗ Proposal rejected. Keeping: {current_structure}')

## Step 4: Full structure descent run (N iterations)

In [ ]:
N_ITERATIONS = 8  # Increase for full experiment

def get_metrics_fn(w, structure=None):
    return compute_metrics(features_list, chosen_indices, w.get_weights, customer_ids, categories)


def _structured_residuals_from_existing(w):
    """Item 2 — structured residual shape for the LLM prompt.

    Groups the same (features_list, chosen_indices, metadata) the old
    compute_residuals call used, then repackages as:
      {"slices": [{name, top1, n_events, pct_first_buy_errors, pct_popular_errors}, ...],
       "overall_patterns": [str, ...]}
    """
    import numpy as _np
    from collections import defaultdict

    # Popularity cutoff (75th percentile) — derived from metadata if available.
    pops = []
    for m in metadata:
        p = m.get('popularity')
        if p is not None:
            try:
                pops.append(float(p))
            except (TypeError, ValueError):
                pass
    pop_p75 = float(_np.quantile(pops, 0.75)) if pops else float('inf')

    slice_stats = defaultdict(lambda: {
        'n': 0, 'correct': 0, 'n_err': 0,
        'first_buy_err': 0, 'popular_err': 0,
    })

    for feats, chosen, cid, cat, meta in zip(
        features_list, chosen_indices, customer_ids, categories, metadata
    ):
        wv = w.get_weights(cid, cat)
        utils = feats @ wv
        is_correct = int(_np.sum(utils > utils[chosen]) == 0)

        s = slice_stats[cat]
        s['n'] += 1
        s['correct'] += is_correct
        if not is_correct:
            s['n_err'] += 1
            routine_val = meta.get('routine', meta.get('is_repeat', None))
            # "first buy" = routine == 0 OR not is_repeat
            if routine_val == 0 or routine_val is False:
                s['first_buy_err'] += 1
            pop_val = meta.get('popularity')
            if pop_val is not None:
                try:
                    if float(pop_val) > pop_p75:
                        s['popular_err'] += 1
                except (TypeError, ValueError):
                    pass

    slices = []
    for cat, s in slice_stats.items():
        if s['n'] == 0:
            continue
        top1 = s['correct'] / s['n']
        n_err = max(s['n_err'], 1)
        slices.append({
            'name': cat,
            'top1': top1,
            'n_events': s['n'],
            'pct_first_buy_errors': s['first_buy_err'] / n_err,
            'pct_popular_errors':   s['popular_err']   / n_err,
        })

    slices.sort(key=lambda d: d['top1'])
    top5 = slices[:5]

    overall_patterns = []
    if top5:
        mean_fb = sum(s['pct_first_buy_errors'] for s in top5) / len(top5)
        mean_pop = sum(s['pct_popular_errors']   for s in top5) / len(top5)
        if mean_fb > 0.4:
            overall_patterns.append(
                'Model accuracy drops sharply on first-time purchases'
            )
        if mean_pop > 0.4:
            overall_patterns.append(
                'Errors concentrate in popular items - popularity may be overweighted'
            )

    return {'slices': top5, 'overall_patterns': overall_patterns}


def get_residuals_fn(w, structure=None):
    return _structured_residuals_from_existing(w)

# ── Accept-strategy selection (ablation knob) ──
from src.accept_strategies import (
    GreedyAccept,
    SimulatedAnnealingAccept,
    ThresholdAccept,
    LateAcceptanceHillClimbing,
)

# Pick ONE for each run — swap to compare.
ACCEPT_STRATEGY = GreedyAccept()
# ACCEPT_STRATEGY = SimulatedAnnealingAccept(T_start=2.0, T_end=0.1)
# ACCEPT_STRATEGY = ThresholdAccept(tau_start=5.0, tau_end=0.0)
# ACCEPT_STRATEGY = LateAcceptanceHillClimbing(history_length=5)

# Resumes from last completed iteration if checkpoint exists
history = run_structure_descent(
    initial_structure=current_structure,
    fit_fn=fit_fn,
    get_metrics_fn=get_metrics_fn,
    get_residuals_fn=get_residuals_fn,
    n_iterations=N_ITERATIONS,
    accept_strategy=ACCEPT_STRATEGY,
    checkpoint=ckpt,
)

from src.dsl import DSLTerm
_terms = history[-1]['structure'].replace('S = ', '').split(' + ')
current_structure = DSLStructure([DSLTerm.parse(t) for t in _terms])
current_score = history[-1]['score']
print(f'\nFinal structure: {current_structure}')
print(f'Final score: {current_score:.2f}')

## Search history

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
display(hist_df)

plt.figure(figsize=(8, 3))
plt.plot(hist_df['iteration'], hist_df['score'], marker='o')
plt.scatter(hist_df[hist_df['accepted']]['iteration'],
            hist_df[hist_df['accepted']]['score'], color='green', zorder=5, label='Accepted')
plt.xlabel('Outer loop iteration')
plt.ylabel('Posterior score')
plt.title('Structure Descent: posterior score over iterations')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save final structure with checkpoint
with open(DATA_DIR / 'final_structure.pkl', 'wb') as f:
    pickle.dump({'structure': current_structure.to_dict(),
                 'score': current_score,
                 'history': history}, f)
ckpt.save('final', {}, ['final_structure.pkl'])
print(f'Saved. Final: {current_structure}  |  Score: {current_score:.2f}')